# ДЗ 2. MLflow: сравнение экспериментов

В этой работе сравниваю два варианта Random Forest для прогноза цены объявления Airbnb. Использую тот же Pipeline и разделение данных, что в ДЗ1. Для каждого запуска записываю в MLflow параметры, метрики на тестовой выборке и обученную модель.

### 1. Установка и импорт библиотек


In [1]:
%pip install -q "mlflow==3.18.0" "boto3==1.43.18" "scikit-learn==1.8.0"

In [2]:
import os
import pickle
import subprocess
import sys
import tempfile
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from mlflow.tracking import MlflowClient
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("MLflow:", mlflow.__version__)
print("scikit-learn:", __import__("sklearn").__version__)

MLflow: 3.18.0
scikit-learn: 1.8.0


### 2. Используем функции из ДЗ1
Чтобы не повторять код подготовки данных, использую функции из train.py.

In [3]:
PROJECT_URL = "https://github.com/kss4891-dev/mlops-airbnb-course.git"

if (Path.cwd() / "train.py").is_file():
    project_dir = Path.cwd()
elif (Path.cwd().parent / "train.py").is_file():
    project_dir = Path.cwd().parent
else:
    project_dir = Path.home() / "mlops-airbnb-course"
    if not (project_dir / "train.py").is_file():
        subprocess.run(
            ["git", "clone", "--depth", "1", PROJECT_URL, str(project_dir)],
            check=True,
        )

sys.path.insert(0, str(project_dir))
from train import make_pipeline, read_s3_data, split_data

print("Используем train.py из:", project_dir)

Используем train.py из: /root/mlops-airbnb-course


### 3. Настройка доступа к S3
Данные загружаю из Yandex Object Storage. Ключи доступа получаю из переменных окружения или Colab Secrets, в сам ноутбук их не добавляю.

In [4]:
KEY_NAMES = ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY")

if not all(os.environ.get(key) for key in KEY_NAMES):
    try:
        from google.colab import userdata
    except ImportError:
        pass
    else:
        for key in KEY_NAMES:
            if not os.environ.get(key):
                value = userdata.get(key)
                if value:
                    os.environ[key] = value

if not all(os.environ.get(key) for key in KEY_NAMES):
    raise RuntimeError(
        "Для чтения S3 нужны AWS_ACCESS_KEY_ID и AWS_SECRET_ACCESS_KEY. "
        "Укажите их через Colab Secrets или переменные окружения."
    )

print("Ключи S3 доступны через переменные окружения (значения скрыты)")

Ключи S3 доступны через переменные окружения (значения скрыты)


### 4. Подключение локального MLflow Tracking


In [5]:
TRACKING_HOME = Path.home() / ".airbnb_mlflow_hw2"
TRACKING_HOME.mkdir(parents=True, exist_ok=True)

tracking_db = (TRACKING_HOME / "mlflow.db").resolve()
artifact_root = (TRACKING_HOME / "artifacts").resolve()
artifact_root.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(f"sqlite:///{tracking_db.as_posix()}")
experiment_name = "airbnb_price_random_forest_hw2"

existing_experiment = mlflow.get_experiment_by_name(experiment_name)
if existing_experiment is None:
    experiment_id = mlflow.create_experiment(
        experiment_name,
        artifact_location=artifact_root.as_uri(),
    )
else:
    experiment_id = existing_experiment.experiment_id

mlflow.set_experiment(experiment_name)
print("Эксперимент:", experiment_name)
print("Experiment ID:", experiment_id)
print("Хранилище MLflow:", TRACKING_HOME)

2026/11/21 20:37:20 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/21 20:37:20 INFO mlflow.store.db.utils: Updating database tables


Эксперимент: airbnb_price_random_forest_hw2
Experiment ID: 1
Хранилище MLflow: /root/.airbnb_mlflow_hw2


### 5. Загрузка данных и разделение на train/test



In [6]:
df = read_s3_data()
X_train, X_test, y_train, y_test = split_data(df)

print("Количество признаков:", X_train.shape[1])
print("Объектов в train:", len(X_train))
print("Объектов в test:", len(X_test))

S3 data loaded: 20151 rows, 13 columns
Train: (16076, 10); test: (4075, 10); shared listings: 0
Количество признаков: 10
Объектов в train: 16076
Объектов в test: 4075


Получилось 20 151 наблюдение. Для обучения использую 16 076 строк, для тестирования 4 075. У модели 10 признаков. Общих listing_id между выборками нет, поэтому одно объявление не попадает одновременно в обучение и тест.

### 6. Два запуска Random Forest с разными гиперпараметрами
В первом запуске оставляю параметры из ДЗ1: 100 деревьев и максимальную глубину 15. Во втором увеличиваю число деревьев до 150, но уменьшаю глубину до 10. Предобработка и разделение на train/test в обоих случаях одинаковые.

In [7]:
configs = [
    {"run_name": "rf_baseline_100_depth15", "n_estimators": 100, "max_depth": 15},
    {"run_name": "rf_variant_150_depth10", "n_estimators": 150, "max_depth": 10},
]

run_ids = []

for config in configs:
    # make_pipeline() возвращает тот же Pipeline, что используется в ДЗ1
    pipeline = make_pipeline()
    pipeline.set_params(
        model__n_estimators=config["n_estimators"],
        model__max_depth=config["max_depth"],
    )

    with mlflow.start_run(run_name=config["run_name"]) as run:
        mlflow.set_tag("homework", "DZ2_MLflow")
        mlflow.set_tag("source", "prepared/listings.csv")

        mlflow.log_params({
            "model_type": "RandomForestRegressor",
            "n_estimators": config["n_estimators"],
            "max_depth": config["max_depth"],
            "random_state": 42,
            "n_jobs": -1,
            "split_method": "GroupShuffleSplit",
            "test_size": 0.2,
            "group_column": "listing_id",
        })

        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        metrics = {
            "test_mae": float(mean_absolute_error(y_test, y_pred)),
            "test_rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
            "test_r2": float(r2_score(y_test, y_pred)),
        }
        mlflow.log_metrics(metrics)

        # Сохраняем полный обученный Pipeline как артефакт этого run
        with tempfile.TemporaryDirectory() as tmpdir:
            model_path = Path(tmpdir) / "model.pkl"
            with model_path.open("wb") as file:
                pickle.dump(pipeline, file, protocol=pickle.HIGHEST_PROTOCOL)
            mlflow.log_artifact(str(model_path), artifact_path="pipeline")

        run_ids.append(run.info.run_id)
        print(
            f"{config['run_name']}: "
            f"MAE={metrics['test_mae']:.2f}, "
            f"RMSE={metrics['test_rmse']:.2f}, "
            f"R²={metrics['test_r2']:.4f}; "
            f"run_id={run.info.run_id}"
        )

assert len(run_ids) == 2 and len(set(run_ids)) == 2
print("Завершено запусков:", len(run_ids))

rf_baseline_100_depth15: MAE=102.48, RMSE=360.77, R²=0.1583; run_id=98ecf5caca4d40659dd982d783f1f64b
rf_variant_150_depth10: MAE=105.54, RMSE=363.04, R²=0.1477; run_id=5657521883f1459aac8ad7766578a0e9
Завершено запусков: 2


### 7. Сравнение результатов через mlflow.search_runs()
С помощью mlflow.search_runs() собираю метрики двух запусков в таблицу и сортирую модели по MAE.

In [8]:
runs = mlflow.search_runs(experiment_ids=[experiment_id])
current_runs = runs.loc[runs["run_id"].isin(run_ids)].copy()

columns = {
    "run_id": "run_id",
    "tags.mlflow.runName": "Модель",
    "params.n_estimators": "Деревья",
    "params.max_depth": "Макс. глубина",
    "metrics.test_mae": "MAE",
    "metrics.test_rmse": "RMSE",
    "metrics.test_r2": "R²",
}
if len(current_runs) != 2:
    raise RuntimeError("MLflow не вернул оба запуска текущей сессии")

comparison = current_runs[list(columns)].rename(columns=columns)
for col in ("Деревья", "Макс. глубина"):
    comparison[col] = pd.to_numeric(comparison[col]).astype(int)
comparison = comparison.sort_values("MAE", ascending=True).reset_index(drop=True)

display(comparison.drop(columns="run_id").round(4))

,Модель,Деревья,Макс. глубина,MAE,RMSE,R²
0,rf_baseline_100_depth15,100,15,102.4813,360.7729,0.1583
1,rf_variant_150_depth10,150,10,105.5427,363.0433,0.1477


По таблице первый вариант работает лучше: MAE 102,48 против 105,54, RMSE 360,77 против 363,04. Значение R² у него тоже выше (0,1583 против 0,1477). Значит, в этом сравнении новые настройки не дали улучшения. Поскольку одновременно изменились два гиперпараметра, по этим результатам нельзя отдельно оценить влияние каждого из них.

### 8. Проверка артефактов и загрузки Pipeline

Проверим, что **каждый** запуск содержит `pipeline/model.pkl`. Затем загрузим артефакт лучшего по MAE запуска и убедимся, что он делает предсказания **без повторного fit**.

In [9]:
client = MlflowClient()
for run_id in run_ids:
    artifacts = client.list_artifacts(run_id, path="pipeline")
    artifact_names = [item.path for item in artifacts]
    assert "pipeline/model.pkl" in artifact_names, (
        f"В запуске {run_id} не найден pipeline/model.pkl: {artifact_names}"
    )
    print("Артефакт найден:", run_id, "pipeline/model.pkl")

best_run_id = comparison.loc[0, "run_id"]
loaded_path = mlflow.artifacts.download_artifacts(
    run_id=best_run_id,
    artifact_path="pipeline/model.pkl",
)
with open(loaded_path, "rb") as file:
    loaded_pipeline = pickle.load(file)

restored_predictions = loaded_pipeline.predict(X_test)
assert len(restored_predictions) == len(X_test)
assert np.isfinite(restored_predictions).all()

print("Лучший run_id:", best_run_id)
print("Предсказаний загруженного Pipeline:", len(restored_predictions))
print("MAE загруженной модели:", round(mean_absolute_error(y_test, restored_predictions), 2))
print("Повторное обучение не выполнялось")

Артефакт найден: 98ecf5caca4d40659dd982d783f1f64b pipeline/model.pkl
Артефакт найден: 5657521883f1459aac8ad7766578a0e9 pipeline/model.pkl


Лучший run_id: 98ecf5caca4d40659dd982d783f1f64b
Предсказаний загруженного Pipeline: 4075
MAE загруженной модели: 102.48
Повторное обучение не выполнялось


Артефакты нашлись у обоих запусков. Загруженная модель выдала 4 075 предсказаний, её MAE составила 102,48. Это совпадает с результатом лучшего запуска с учётом округления. Повторно обучать модель для получения прогноза не потребовалось.

### 9. Вывод по экспериментам

Сравнение выполнялось на одном датасете, при одинаковом разделении на train/test и одинаковой предобработке. Между двумя запусками менялись только `n_estimators` и `max_depth`. В MLflow сохранены параметры, тестовые метрики и `model.pkl` каждого запуска.

In [10]:
best = comparison.iloc[0]
baseline_mae = float(
    comparison.loc[comparison["Модель"] == "rf_baseline_100_depth15", "MAE"].iloc[0]
)
best_mae = float(best["MAE"])

print(
    f"По MAE лучший результат показала модель {best['Модель']}: "
    f"MAE={best_mae:.2f}, RMSE={best['RMSE']:.2f}, R²={best['R²']:.4f}."
)
if best_mae < baseline_mae:
    print(f"По сравнению с вариантом из ДЗ1 MAE уменьшилась на {baseline_mae-best_mae:.2f}.")
else:
    print("Вариант из ДЗ1 по MAE не уступил альтернативному запуску.")
print("Оба запуска зарегистрированы в MLflow, обученные Pipeline сохранены как артефакты.")

По MAE лучший результат показала модель rf_baseline_100_depth15: MAE=102.48, RMSE=360.77, R²=0.1583.
Вариант из ДЗ1 по MAE не уступил альтернативному запуску.
Оба запуска зарегистрированы в MLflow, обученные Pipeline сохранены как артефакты.


В результате лучше показала себя исходная конфигурация Random Forest из ДЗ1. Увеличение количества деревьев при одновременном уменьшении глубины не улучшило качество на нашей тестовой выборке. При этом R² остаётся невысоким у обеих моделей, поэтому качество прогноза ещё можно улучшать.

MLflow сохранил параметры, метрики и model.pkl для каждого запуска. Я также проверила, что сохранённый Pipeline можно загрузить и использовать для предсказаний без повторного обучения.